# ボロノイ領域の時系列変化（試合単位）

- 1パネル = 1試合、6選手のボロノイ領域面積の折れ線グラフ
- 3×3グリッド（7パネル使用）をセッション単位で出力
- 4ゲーム × 3セッション = 計12画像

In [ ]:
import sys
from pathlib import Path

src_dir = str(Path().resolve().parents[1])
if src_dir not in sys.path:
    sys.path.append(src_dir)

In [ ]:
from analyzers.datamanager import DataManager
from analyzers import utils, calculator

data_manager = DataManager("../processed_data")
all_data = data_manager.get_all_trajectories()

# G1-G4 のみ対象にして y軸・x軸の上限を統一
g_data = {k: v for k, v in all_data.items() if k.startswith('G')}
max_value = calculator.max_recursive(g_data)
graph_ylim = utils.round_up(max_value / 10000.0)  # m² に変換
max_len = calculator.max_len_recursive(g_data)
graph_xlim = max_len

print(f"Y-axis limit: {graph_ylim} m²")
print(f"X-axis limit: {graph_xlim} frames")

In [ ]:
from analyzers.voronoi_timeseries.event_loader import get_first_catch_frame, build_event_path
from analyzers.voronoi_timeseries.drawer import plot_session_grid

DATASET_DIR = Path().resolve().parents[2] / "_dataset_yamaha"
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for game_num in range(1, 5):
    match_type = f"G{game_num}"
    for session_num in range(1, 4):
        trials_data = []
        for trial_num in range(1, 8):
            game_name = f"basket_G{game_num}-S{session_num}T{trial_num}"
            player_data = all_data[match_type][game_name]

            event_path = build_event_path(str(DATASET_DIR), game_num, session_num, trial_num)
            catch_frame = get_first_catch_frame(event_path)

            trials_data.append({
                'player_data': player_data,
                'catch_frame': catch_frame,
                'title': f"T{trial_num} ({game_name})",
            })

        save_path = RESULTS_DIR / f"voronoi_timeseries_G{game_num}_S{session_num}.png"
        plot_session_grid(
            trials_data=trials_data,
            ylim=graph_ylim,
            xlim=graph_xlim,
            save_path=str(save_path),
        )
        print(f"Saved: {save_path}")